In [13]:
import geopandas as gpd

In [14]:
city = "ACC"
num_votes = "860k"
user1 = 'planners'
user2 = 'communities'
# Load the two GPKG files
gdf1 = gpd.read_file(f'{city}/hotspot/n_ts/{city}_n_ts_hotspot_{user1}_wtsm_{num_votes}.gpkg')
gdf2 = gpd.read_file(f'{city}/hotspot/n_ts/{city}_n_ts_hotspot_{user2}_wtsm_{num_votes}.gpkg')
out_gdf = f'{city}/hotspot/diff/{city}_n_ts_hotspot_match_{user1}_{user2}_{num_votes}.gpkg'

In [19]:
print(gdf1['cluster_type'].unique())
print(gdf2['cluster_type'].unique())

['Not Significant' 'LL' 'HH' 'HL' 'LH']
['Not Significant' 'HL' 'LL' 'LH' 'HH']


In [20]:
# Merge the two dataframes on the shared column "PARTIMAP_I"
merged = gdf1[['PARTIMAP_I', 'cluster_type', 'geometry']].merge(
        gdf2[['PARTIMAP_I', 'cluster_type']],
        on='PARTIMAP_I',
        suffixes=('_1', '_2')
)

# Define the comparison logic
def compare_cluster_types(row):
    if ((row['cluster_type_1'] == "HH") & (row['cluster_type_2']=="HH")):
        return 4  # Both have "HH"
    elif ((row['cluster_type_1'] == "LL") & (row['cluster_type_2']=="LL")):
        return -4 # Both have "LL"
    elif (row['cluster_type_1'] == 'HH' and row['cluster_type_2'] == 'LL'):
        return 1 # user1 has HH and user2 has LL
    elif (row['cluster_type_1'] == 'LL' and row['cluster_type_2'] == 'HH'):
        return -1  # user1 has LL and user2 has HH
    elif (row['cluster_type_1'] == 'HH' and (row['cluster_type_2'] != 'HH' or row['cluster_type_2'] != 'LL')):
        return 3
    elif ((row['cluster_type_1'] != 'HH' or row['cluster_type_2'] != 'LL') and row['cluster_type_2'] == 'HH'):
        return 2
    elif (row['cluster_type_1'] == 'LL' and (row['cluster_type_2'] != 'HH' or row['cluster_type_2'] != 'LL')):
        return -3
    elif ((row['cluster_type_1'] != 'HH' or row['cluster_type_2'] != 'LL') and row['cluster_type_2'] == 'LL'):
        return -2
    else:
        return 0  # Any other case (e.g., one is missing or has a different value)

# Apply the comparison logic
merged['compare'] = merged.apply(compare_cluster_types, axis=1)

output_gdf = merged[['PARTIMAP_I', 'compare', 'geometry']]

# Save the result to a new GPKG file
output_gdf.to_file(out_gdf, driver='GPKG')